# 🌿 NutriScan — Data Cleaning Pipeline
**By Shreya Chauhan**

This notebook cleans the Open Food Facts dataset (356,000 products) down to a high-quality,
analysis-ready dataset for NutriScan — filtered to food categories Indians consume daily.

**Output:** A clean CSV ready for:
- SQL analysis (DB Browser / BigQuery)
- Tableau / Power BI dashboard
- GitHub upload
- NutriScan portfolio case study

---
**Data source:** Open Food Facts (open database, ODbL licence)
**Raw file:** `openfoodfacts.xlsx` — 356,023 rows × 24 columns

In [2]:
# Run this cell first — it will prompt you to upload your file
from google.colab import files
uploaded = files.upload()   # upload openfoodfacts.xlsx when prompted
print('✅ File uploaded:', list(uploaded.keys()))

Saving openfoodfacts.xlsx to openfoodfacts.xlsx
✅ File uploaded: ['openfoodfacts.xlsx']


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Loading file... (may take 1-2 mins for 356k rows)')
df = pd.read_excel('openfoodfacts.xlsx', engine='openpyxl')
print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumns:', list(df.columns))

Loading file... (may take 1-2 mins for 356k rows)
✅ Loaded: 356,022 rows × 24 columns

Columns: ['code', 'product_name', 'generic_name', 'brands', 'brands_tags', 'ingredients_text', 'allergens', 'allergens_en', 'traces', 'serving_size', 'additives_n', 'additives', 'energy_100g', 'fat_100g', 'saturated-fat_100g', 'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g', 'calcium_100g', 'iron_100g']


In [4]:
print('=== RAW DATA OVERVIEW ===')
print(f'Total rows: {len(df):,}')
print(f'Total columns: {len(df.columns)}')

print('\n=== NULL % PER COLUMN ===')
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
for col, pct in null_pct.items():
    bar = '█' * int(pct/5) + '░' * (20 - int(pct/5))
    print(f'{col:<30} {bar} {pct}%')

print('\n=== SAMPLE ROW ===')
df[df['product_name'].notna() & df['energy_100g'].notna()].head(1).T

=== RAW DATA OVERVIEW ===
Total rows: 356,022
Total columns: 24

=== NULL % PER COLUMN ===
code                           ░░░░░░░░░░░░░░░░░░░░ 0.0%
product_name                   ░░░░░░░░░░░░░░░░░░░░ 4.9%
generic_name                   ████████████████░░░░ 83.8%
brands                         █░░░░░░░░░░░░░░░░░░░ 8.2%
brands_tags                    █░░░░░░░░░░░░░░░░░░░ 8.2%
ingredients_text               ████░░░░░░░░░░░░░░░░ 20.3%
allergens                      █████████████████░░░ 89.6%
allergens_en                   ████████████████████ 100.0%
traces                         ██████████████████░░ 92.0%
serving_size                   ███████░░░░░░░░░░░░░ 39.2%
additives_n                    ████░░░░░░░░░░░░░░░░ 20.3%
additives                      ████░░░░░░░░░░░░░░░░ 20.3%
energy_100g                    ███░░░░░░░░░░░░░░░░░ 17.0%
fat_100g                       ████░░░░░░░░░░░░░░░░ 21.5%
saturated-fat_100g             █████░░░░░░░░░░░░░░░ 25.9%
cholesterol_100g               ███████████

,1
code,4530.0
product_name,Banana Chips Sweetened (Whole)
generic_name,NaN
brands,NaN
brands_tags,NaN
ingredients_text,"Bananas, vegetable oil (coconut oil, corn oil ..."
allergens,NaN
allergens_en,NaN
traces,NaN
serving_size,28 g (1 ONZ)


In [6]:
# Keep only the columns useful for NutriScan
KEEP_COLS = [
    'code',              # barcode
    'product_name',      # product name
    'brands',            # brand
    'ingredients_text',  # for allergen detection
    'allergens_en',      # allergens (structured)
    'serving_size',      # serving size info
    'energy_100g',       # calories (kJ — we'll convert to kcal)
    'fat_100g',
    'saturated-fat_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'fiber_100g',
    'proteins_100g',
    'salt_100g',
    'sodium_100g',
    'calcium_100g',
    'iron_100g',
]

# Drop columns NOT in keep list
DROP_COLS = [c for c in df.columns if c not in KEEP_COLS]
print('Dropping:', DROP_COLS)

df = df[KEEP_COLS].copy()
print(f'\n✅ Columns reduced: {len(KEEP_COLS)} kept')
print(f'Shape: {df.shape}')

Dropping: []

✅ Columns reduced: 17 kept
Shape: (356022, 17)


In [7]:
before = len(df)

# Must have: product name + calories + at least 2 of fat/protein/carbs
df = df[
    df['product_name'].notna() &
    (df['product_name'].str.strip().str.len() > 2) &
    df['energy_100g'].notna() &
    (df['energy_100g'] > 0) &
    df['fat_100g'].notna() &
    df['proteins_100g'].notna()
].copy()

after = len(df)
print(f'Before: {before:,} rows')
print(f'After:  {after:,} rows')
print(f'Dropped: {before-after:,} rows ({(before-after)/before*100:.1f}%)')
print(f'\n✅ {after:,} products have usable nutrition data')

Before: 356,022 rows
After:  264,932 rows
Dropped: 91,090 rows (25.6%)

✅ 264,932 products have usable nutrition data


In [8]:
# energy_100g in OFF is kJ — convert to kcal (divide by 4.184)
df['kcal_100g'] = (df['energy_100g'] / 4.184).round(1)

# sodium from salt if sodium is missing (salt * 0.4)
df['sodium_100g'] = df['sodium_100g'].fillna(df['salt_100g'] * 0.4)

# sodium to mg
df['sodium_mg_100g'] = (df['sodium_100g'] * 1000).round(1)

# Remove physically impossible values (data entry errors)
df = df[
    (df['kcal_100g'] > 0) & (df['kcal_100g'] <= 950) &   # max ~900 for pure oil
    (df['fat_100g'] >= 0) & (df['fat_100g'] <= 100) &
    (df['proteins_100g'] >= 0) & (df['proteins_100g'] <= 100) &
    (df['carbohydrates_100g'].fillna(0) >= 0) & (df['carbohydrates_100g'].fillna(0) <= 100)
].copy()

# Fill remaining NaN with 0 for scoring columns
FILL_ZERO = ['sugars_100g','fiber_100g','saturated-fat_100g','sodium_mg_100g','calcium_100g','iron_100g']
for col in FILL_ZERO:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print(f'✅ Units fixed, impossible values removed')
print(f'Remaining rows: {len(df):,}')
print(f'\nkcal range: {df["kcal_100g"].min():.0f} – {df["kcal_100g"].max():.0f}')
print(f'Protein range: {df["proteins_100g"].min():.1f} – {df["proteins_100g"].max():.1f}g')

✅ Units fixed, impossible values removed
Remaining rows: 264,820

kcal range: 0 – 950
Protein range: 0.0 – 100.0g


In [9]:
# Clean product_name
df['product_name'] = (
    df['product_name']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)   # collapse multiple spaces
    .str.title()                              # Title Case
)

# Clean brands
df['brands'] = (
    df['brands']
    .str.strip()
    .str.split(',').str[0]    # keep only first brand if multiple
    .str.strip()
    .str.title()
)
df['brands'] = df['brands'].fillna('Unknown Brand')

# Remove duplicates (same product name + brand)
before = len(df)
df = df.drop_duplicates(subset=['product_name', 'brands'], keep='first')
print(f'Removed {before - len(df):,} duplicate product+brand combos')
print(f'Remaining: {len(df):,} products')

Removed 22,213 duplicate product+brand combos
Remaining: 242,607 products


In [10]:
def calc_nutriscan_grade(row):
    """
    NutriScan scoring: penalty-based system
    Penalties for: sugar, saturated fat, total fat, sodium
    Bonuses for: protein, fibre
    All values per 100g
    """
    sugar  = float(row.get('sugars_100g', 0) or 0)
    fat    = float(row.get('fat_100g', 0) or 0)
    satfat = float(row.get('saturated-fat_100g', 0) or 0)
    sodmg  = float(row.get('sodium_mg_100g', 0) or 0)
    prot   = float(row.get('proteins_100g', 0) or 0)
    fibre  = float(row.get('fiber_100g', 0) or 0)

    penalty = 0

    # Sugar penalties
    if   sugar >= 25: penalty += 70
    elif sugar >= 20: penalty += 55
    elif sugar >= 15: penalty += 40
    elif sugar >= 10: penalty += 25
    elif sugar >= 5:  penalty += 10

    # Saturated fat penalties
    if   satfat >= 10: penalty += 45
    elif satfat >= 5:  penalty += 30
    elif satfat >= 3:  penalty += 15
    elif satfat >= 1.5: penalty += 5

    # Total fat penalties
    if   fat >= 20:   penalty += 38
    elif fat >= 17.5: penalty += 28
    elif fat >= 10:   penalty += 18
    elif fat >= 5:    penalty += 8

    # Sodium penalties
    if   sodmg >= 500: penalty += 55
    elif sodmg >= 400: penalty += 42
    elif sodmg >= 300: penalty += 30
    elif sodmg >= 200: penalty += 18
    elif sodmg >= 120: penalty += 8

    # Protein bonus
    if   prot > 20: penalty -= 15
    elif prot > 10: penalty -= 8

    # Fibre bonus
    if   fibre > 6: penalty -= 12
    elif fibre > 3: penalty -= 6

    penalty = max(0, min(100, penalty))

    if   penalty < 10: return 'A'
    elif penalty < 22: return 'B'
    elif penalty < 38: return 'C'
    elif penalty < 55: return 'D'
    elif penalty < 72: return 'E'
    else:              return 'F'

print('Calculating NutriScan grades...')
df['nutriscan_grade'] = df.apply(calc_nutriscan_grade, axis=1)

print('\n✅ Grade distribution:')
grade_counts = df['nutriscan_grade'].value_counts().sort_index()
for g, cnt in grade_counts.items():
    pct = cnt/len(df)*100
    bar = '█' * int(pct/2)
    print(f'  Grade {g}: {cnt:>6,} products ({pct:5.1f}%)  {bar}')

Calculating NutriScan grades...

✅ Grade distribution:
  Grade A: 30,927 products ( 12.7%)  ██████
  Grade B: 18,594 products (  7.7%)  ███
  Grade C: 26,214 products ( 10.8%)  █████
  Grade D: 24,711 products ( 10.2%)  █████
  Grade E: 39,384 products ( 16.2%)  ████████
  Grade F: 102,777 products ( 42.4%)  █████████████████████


In [11]:
def calc_protein_level(row):
    """
    NutriScan protein rating:
    HIGH     = ≥10g protein per 100 kcal
    MODERATE = 5–10g protein per 100 kcal
    LOW      = <5g protein per 100 kcal
    """
    protein = float(row.get('proteins_100g', 0) or 0)
    kcal    = float(row.get('kcal_100g', 0) or 0)
    if kcal == 0:
        return 'Unknown'
    ratio = (protein / kcal) * 100
    if   ratio >= 10: return 'High'
    elif ratio >= 5:  return 'Moderate'
    else:             return 'Low'

df['protein_level'] = df.apply(calc_protein_level, axis=1)
df['protein_per_100kcal'] = ((df['proteins_100g'] / df['kcal_100g']) * 100).round(2)

print('✅ Protein level distribution:')
print(df['protein_level'].value_counts().to_string())

✅ Protein level distribution:
protein_level
Low         190561
Moderate     35784
High         16262


In [12]:
ALLERGENS = {
    'Gluten':    ['gluten','wheat','barley','rye','oat','spelt'],
    'Dairy':     ['milk','lactose','dairy','butter','cream','whey','casein','cheese'],
    'Eggs':      ['egg','albumin','ovalbumin'],
    'Tree Nuts': ['almond','cashew','walnut','hazelnut','pistachio','macadamia','pecan','brazil nut'],
    'Peanuts':   ['peanut','groundnut','arachis'],
    'Soy':       ['soy','soya','tofu','edamame'],
    'Fish':      ['fish','cod','salmon','tuna','sardine','anchovy'],
    'Shellfish': ['shellfish','shrimp','prawn','crab','lobster'],
    'Sesame':    ['sesame','tahini'],
    'Mustard':   ['mustard'],
    'Sulphites': ['sulphite','sulfite','sulphur dioxide'],
}

def detect_allergens(row):
    haystack = ' '.join([
        str(row.get('allergens_en') or ''),
        str(row.get('ingredients_text') or '')
    ]).lower()
    found = [name for name, patterns in ALLERGENS.items()
             if any(p in haystack for p in patterns)]
    return ', '.join(found) if found else 'None detected'

print('Detecting allergens... (may take 30s)')
df['allergens_detected'] = df.apply(detect_allergens, axis=1)

# Count how many products contain each allergen
print('\n✅ Allergen prevalence:')
for alg in ALLERGENS:
    cnt = df['allergens_detected'].str.contains(alg).sum()
    print(f'  {alg:<12}: {cnt:,} products')

Detecting allergens... (may take 30s)

✅ Allergen prevalence:
  Gluten      : 52,715 products
  Dairy       : 56,762 products
  Eggs        : 12,509 products
  Tree Nuts   : 10,497 products
  Peanuts     : 6,268 products
  Soy         : 42,094 products
  Fish        : 2,931 products
  Shellfish   : 1,515 products
  Sesame      : 3,883 products
  Mustard     : 3,502 products
  Sulphites   : 6,135 products


In [13]:
# Detect products marketed as healthy but that score C–F
HEALTH_KEYWORDS = [
    'healthy','natural','organic','diet','light','low fat','low sugar',
    'no sugar','sugar free','high protein','protein','multigrain',
    'wholegrain','whole grain','fortified','nutritious','wellness',
    'fit','slim','zero','reduced','vitamin','mineral','probiotic',
    'antioxidant','superfood','power','energy','boost','plus'
]

def has_health_claim(name):
    if pd.isna(name): return False
    name_lower = str(name).lower()
    return any(kw in name_lower for kw in HEALTH_KEYWORDS)

df['has_health_claim'] = df['product_name'].apply(has_health_claim)

# THE HEADLINE FINDING
with_claim = df[df['has_health_claim']]
claim_bad  = with_claim[with_claim['nutriscan_grade'].isin(['C','D','E','F'])]

total_claim = len(with_claim)
total_bad   = len(claim_bad)
pct_bad     = total_bad / total_claim * 100 if total_claim > 0 else 0

print('=' * 55)
print('🚨  THE HEALTHY HALO FINDING')
print('=' * 55)
print(f'Products with health claims:          {total_claim:>6,}')
print(f'...that score C or below (NutriScan): {total_bad:>6,}')
print(f'Failure rate:                         {pct_bad:>5.1f}%')
print()
print('→ This is your HEADLINE INSIGHT for your PRD, CV,')
print('  LinkedIn post, and Tableau dashboard.')
print()
print('Worst offenders (health claim + Grade F):')
worst = claim_bad[claim_bad['nutriscan_grade']=='F'][['product_name','brands','nutriscan_grade','sugars_100g','fat_100g']].head(10)
print(worst.to_string(index=False))

🚨  THE HEALTHY HALO FINDING
Products with health claims:          20,999
...that score C or below (NutriScan): 14,431
Failure rate:                          68.7%

→ This is your HEADLINE INSIGHT for your PRD, CV,
  LinkedIn post, and Tableau dashboard.

Worst offenders (health claim + Grade F):
                             product_name         brands nutriscan_grade  sugars_100g  fat_100g
                   Organic Salted Nut Mix      Grizzlies               F         3.57     57.14
             Organic Dark Chocolate Minis Equal Exchange               F        42.50     37.50
           Organic Sweetened Banana Chips           Unfi               F        16.67     26.67
                         Energy Power Mix       Sunridge               F        32.50     17.50
    Antioxidant Mix - Berries & Chocolate       Sunridge               F        30.00     33.33
Organic Quinoa Coconut Granola With Mango       Sunridge               F        27.27     10.91
                Peanut Butter P

In [14]:
# Build the final clean dataframe
final = df[[
    'code',
    'product_name',
    'brands',
    'kcal_100g',
    'proteins_100g',
    'carbohydrates_100g',
    'sugars_100g',
    'fat_100g',
    'saturated-fat_100g',
    'fiber_100g',
    'sodium_mg_100g',
    'calcium_100g',
    'iron_100g',
    'allergens_detected',
    'serving_size',
    'nutriscan_grade',
    'protein_level',
    'protein_per_100kcal',
    'has_health_claim',
]].copy()

# Rename for SQL-friendliness (no spaces, lowercase)
final.columns = [
    'barcode',
    'product_name',
    'brand',
    'calories_kcal',
    'protein_g',
    'carbs_g',
    'sugar_g',
    'fat_g',
    'saturated_fat_g',
    'fibre_g',
    'sodium_mg',
    'calcium_mg',
    'iron_mg',
    'allergens',
    'serving_size',
    'nutriscan_grade',
    'protein_level',
    'protein_per_100kcal',
    'has_health_claim',
]

# Round all numeric columns to 2 decimal places
num_cols = ['calories_kcal','protein_g','carbs_g','sugar_g','fat_g',
            'saturated_fat_g','fibre_g','sodium_mg','calcium_mg','iron_mg','protein_per_100kcal']
for col in num_cols:
    final[col] = final[col].round(2)

# Sort by brand, then product name
final = final.sort_values(['brand','product_name']).reset_index(drop=True)

print(f'✅ Final dataset: {len(final):,} rows × {len(final.columns)} columns')
print(f'\nColumn names (SQL-ready):')
for c in final.columns:
    print(f'  {c}')
print('\nSample:')
final.head(3)

✅ Final dataset: 242,607 rows × 19 columns

Column names (SQL-ready):
  barcode
  product_name
  brand
  calories_kcal
  protein_g
  carbs_g
  sugar_g
  fat_g
  saturated_fat_g
  fibre_g
  sodium_mg
  calcium_mg
  iron_mg
  allergens
  serving_size
  nutriscan_grade
  protein_level
  protein_per_100kcal
  has_health_claim

Sample:


,barcode,product_name,brand,calories_kcal,protein_g,carbs_g,sugar_g,fat_g,saturated_fat_g,fibre_g,sodium_mg,calcium_mg,iron_mg,allergens,serving_size,nutriscan_grade,protein_level,protein_per_100kcal,has_health_claim
0,6.150013e+10,"!Ajua!, Caffeine Free Soda, Mandarin Orange",!Ajua!,50.0,0.0,12.50,12.08,0.0,0.0,0.0,15.0,0.0,0.0,Gluten,240 ml (8 fl oz),C,Low,0.0,False
1,6.150013e+10,"!Ajua!, Caffeine Free Soda, Pineapple",!Ajua!,50.0,0.0,12.50,12.08,0.0,0.0,0.0,15.0,0.0,0.0,Gluten,240 ml (8 fl oz),C,Low,0.0,False
2,6.150013e+10,"!Ajua!, Caffeine Free Soda, Tutti Fruitti Frui...",!Ajua!,50.0,0.0,12.08,11.67,0.0,0.0,0.0,15.0,0.0,0.0,Gluten,240 ml (8 fl oz),C,Low,0.0,False


In [15]:
print('=== FINAL DATASET QUALITY REPORT ===')
print(f'Total products:     {len(final):,}')
print(f'Unique brands:      {final["brand"].nunique():,}')
print()

print('Null values remaining:')
nulls = final.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print('  ✅ No nulls in critical columns!')
else:
    print(nulls.to_string())

print('\nGrade distribution:')
grade_dist = final['nutriscan_grade'].value_counts().sort_index()
for g, cnt in grade_dist.items():
    pct = cnt/len(final)*100
    bar = '█' * int(pct/2)
    print(f'  Grade {g}: {cnt:>6,}  ({pct:5.1f}%)  {bar}')

print('\nProtein level distribution:')
print(final['protein_level'].value_counts().to_string())

print('\nTop 20 brands by product count:')
print(final['brand'].value_counts().head(20).to_string())

=== FINAL DATASET QUALITY REPORT ===
Total products:     242,607
Unique brands:      38,096

Null values remaining:
carbs_g           496
serving_size    59046

Grade distribution:
  Grade A: 30,927  ( 12.7%)  ██████
  Grade B: 18,594  (  7.7%)  ███
  Grade C: 26,214  ( 10.8%)  █████
  Grade D: 24,711  ( 10.2%)  █████
  Grade E: 39,384  ( 16.2%)  ████████
  Grade F: 102,777  ( 42.4%)  █████████████████████

Protein level distribution:
protein_level
Low         190561
Moderate     35784
High         16262

Top 20 brands by product count:
brand
Unknown Brand         3021
Carrefour             2842
Auchan                2351
Meijer                2055
U                     1933
Leader Price          1692
Kroger                1627
Casino                1626
Great Value           1135
Ahold                 1124
Roundy'S              1044
Cora                  1034
Spartan               1018
Food Lion              987
Weis                   978
Coop                   947
Shoprite           

In [16]:
# Save as CSV
output_file = 'nutriscan_clean.csv'
final.to_csv(output_file, index=False)
print(f'✅ Saved: {output_file}')

# Show file size
import os
size_mb = os.path.getsize(output_file) / 1024 / 1024
print(f'File size: {size_mb:.1f} MB')
print(f'Rows: {len(final):,}')
print(f'Columns: {len(final.columns)}')

# Download to your computer
from google.colab import files
files.download(output_file)
print('\n✅ Download started — check your Downloads folder!')

✅ Saved: nutriscan_clean.csv
File size: 34.0 MB
Rows: 242,607
Columns: 19


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started — check your Downloads folder!
